# 00 pipeline 自动配置

目标：用 `pipeline("text-generation")` 体验最高层封装，观察它如何自动处理 tokenizer、model、generate 和 decode。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


## 1. 加载 pipeline

`pipeline` 是最省事的入口，适合快速验证模型和提示词。


In [ ]:
from transformers import pipeline


def print_pipeline_answer(result):
    generated_text = result[0]["generated_text"]

    if isinstance(generated_text, list):
        print(generated_text[-1]["content"])
    else:
        print(generated_text)


pipe = pipeline(
    "text-generation",
    model=MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)


## 2. 运行一次生成

这里把输入保持为 messages 格式，让 tokenizer 自动套用 chat template。


In [ ]:
messages = [
    {
        "role": "system",
        "content": "你是个描述大师，负责帮助用户生成相关描述画面，请扩写",
    },
    {
        "role": "user",
        "content": "一只熊在森林里",
    }
]

result = pipe(
    messages,
    max_new_tokens=1024,
    do_sample=False,
)

print_pipeline_answer(result)
